<div style="font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif;max-width:900px;margin:0 auto;border-radius:16px;overflow:hidden;box-shadow:0 4px 20px rgba(0,0,0,0.12);border:1px solid #e2e2e2;">

<div style="background:linear-gradient(135deg,#002855 0%,#004b8d 55%,#0077c8 100%);padding:28px 20px 22px 20px;text-align:center;">
<img src="./img/ITESOLogo.png" alt="ITESO" width="260" style="margin-bottom:10px;">
<div style="color:#ffffff;font-size:15px;font-weight:600;letter-spacing:0.5px;text-transform:uppercase;opacity:0.9;">
Departamento de Electrónica, Sistemas e Informática
</div>
</div>

<div style="background-color:#ffffff;padding:26px 30px 30px 30px;text-align:center;">

<div style="color:#002855;font-size:26px;font-weight:800;margin-bottom:6px;">
Big Data Analysis
</div>

<div style="display:inline-block;background-color:#eaf4fb;color:#0077c8;font-size:13px;font-weight:700;padding:4px 14px;border-radius:20px;letter-spacing:0.5px;margin-bottom:22px;">
Autumn 2026
</div>


<hr style="border:none;border-top:2px solid #f0f0f0;margin:0 0 22px 0;">

<div style="background-color:#f7fafd;border-left:5px solid #0077c8;border-radius:8px;padding:14px 18px;text-align:left;margin-bottom:18px;">
<div style="font-size:12px;color:#7a7a7a;font-weight:600;text-transform:uppercase;letter-spacing:0.5px;margin-bottom:4px;">
Midterm exam
</div>
<div style="font-size:19px;color:#002855;font-weight:700;">
Batch Processing
</div>
</div>

<div style="font-size:14px;color:#444;margin-top:20px;">
<span style="font-weight:700;color:#002855;">Profesor:</span> Pablo Camarillo Ramírez
</div>

</div>
</div>

**Name:** 

### Rules
- Work in this notebook. Import **your own** `SparkUtils` class from the `src` directory (Lab 04).
- Use the **DataFrame API** (`pyspark.sql.functions`).
- Every *Written answer* cell must be answered in your own words. Code without written answers will not receive full credit.
- Export the notebook as **PDF** (first HTML, then PDF) and upload it to Canvas.

### Dataset — `food_delivery/`
A food-delivery platform exported its January and February orders in **two separate batches**, plus two catalog files whose descriptive fields are stored as **JSON strings**.

| File | Columns |
|---|---|
| `orders_2026_01.csv` | order_id, restaurant_id, courier_id, customer_city, order_ts, items, order_total, delivery_minutes, rating, payment_info *(JSON)*, _batch_id, _debug |
| `orders_2026_02.csv` | same columns, **but in a different order** — inspect the header |
| `restaurants.csv` | restaurant_id, restaurant_info *(JSON)* |
| `couriers.csv` | courier_id, courier_info *(JSON)* |

JSON examples:
```
restaurant_info: {"name": "Sushi Kaze", "cuisine": "Japanese", "location": {"city": "Zapopan", "zone": "Andares"}, "price_level": 3}
courier_info:    {"courier_name": "Ana Torres", "vehicle": "Bicycle", "contract": {"type": "Part-time"}}
payment_info:    {"method": "wallet", "promo": {"code": "FIRST20", "discount_pct": 20}}   # "promo" is optional
```

---
## Part 1 — Ingestion and Union (15 pts)

**1.1 (5 pts)** Create the Spark session with your `SparkUtils` class and define the schemas of the **four** files with `SparkUtils.generate_schema`. Read JSON columns as `string`, `order_ts` as `timestamp`, money/ratings as `double`.

**1.2 (5 pts)** Read the four CSV files using `header` and the right `escape` option (the JSON contains double quotes).

**1.3 (5 pts)** Combine both monthly files into a single DataFrame called `orders_df`.
✅ *Sanity check:* `orders_df` must have **80 rows** and `order_id` **O0041** must have `customer_city = Monterrey` and `restaurant_id = 3`.

In [ ]:
# Your code here


In [ ]:
# Your code here


In [ ]:
# Your code here


**Written answer 1.3** — Why is `unionByName` the correct choice here instead of `union`? Describe concretely what would have gone wrong with `union`.

*Your answer here.*

---
## Part 2 — Cleaning (20 pts)

Apply the following steps to `orders_df` and store the result in `clean_df`:

1. Drop the internal columns `_batch_id` and `_debug`.
2. Drop rows missing `customer_city`.
3. Compute the average `rating` (ignoring nulls) **with an aggregation**, round it to 2 decimals, and use it to fill the missing ratings.
4. Fill missing `order_total` with `0.0` — those orders were **cancelled**.

Print the null count per column **before** and **after**, and the row count after each step.
✅ *Sanity check:* **80 → 74 rows** and **0 nulls** in every column.

In [ ]:
# Your code here


**Written answer 2** — Filling `order_total` with `0.0` keeps cancelled orders in the dataset. What side-effect does this have on an *average order value* metric, and how should you compute that metric correctly later?

*Your answer here.*

---
## Part 3 — Analysis with the DataFrame API (25 pts)

**3.1 (7 pts)** Using `filter` and `count`: how many **cancelled** orders (`order_total == 0`) are there, and how many **completed late deliveries** (`order_total > 0` **and** `delivery_minutes > 45`)? Express the late deliveries as a percentage of the completed orders.

**3.2 (8 pts)** Using `orderBy`, show the **5 orders with the highest `order_total`** (`order_id, customer_city, restaurant_id, items, order_total, delivery_minutes`).

**3.3 (10 pts)** Considering **only completed orders**, compute per `customer_city`: number of orders (`n_orders`), average ticket (`avg_ticket`, 2 decimals) and average delivery time (`avg_minutes`, 1 decimal). Sort by `n_orders` descending, then `customer_city` ascending.

In [ ]:
# Your code here


In [ ]:
# Your code here


**Written answer 3.2** — One order stands out from the rest of the top 5. Identify it and give **two** plausible explanations (one data-quality, one business).

*Your answer here.*

In [ ]:
# Your code here


**Written answer 3.3** — Which city has the highest average ticket? Is that result trustworthy given what you found in 3.2? Justify with the numbers.

*Your answer here.*

---
## Part 4 — JSON Extraction and Joins (25 pts)

**4.1 (10 pts)** Extract the following columns with `get_json_object` (or `from_json`):

| DataFrame | New columns (JSON path) |
|---|---|
| `restaurants_df` | `restaurant_name` (`$.name`), `cuisine`, `restaurant_city` (`$.location.city`) |
| `couriers_df` | `courier_name`, `vehicle`, `contract_type` (`$.contract.type`) |
| `clean_df` | `payment_method` (`$.method`), `promo_code` (`$.promo.code`) |

**4.2 (10 pts)** Join `clean_df` → `restaurants_df` (on `restaurant_id`) → `couriers_df` (on `courier_id`) with **inner** joins, keeping only completed orders.

**4.3 (5 pts)** From the joined DataFrame, build a report per `cuisine` with `n_orders`, `revenue` (sum of `order_total`, 2 decimals) and `avg_minutes` (1 decimal), sorted by `revenue` descending.

⚠️ **Constraint:** cells 4.1–4.3 may contain **only one action** — the final `show()`.

In [ ]:
# Your code here


---
## Part 5 — Understanding Checks (15 pts)

**5.1 (5 pts) Lazy evaluation.** In Part 4 you wrote ~15 transformations before calling `show()`. What had Spark actually done *before* `show()`? What did `show()` trigger? Why is this beneficial?

**5.2 (5 pts) Execution plan.** Run `.explain()` on `city_stats_df` (Part 3.3). Identify the operator that represents the **shuffle**, say which of your transformations caused it (narrow vs. wide), and explain why `HashAggregate` appears **twice**.

**5.3 (5 pts) Join types.** Count the rows of `clean_df` (completed orders only) joined with `restaurants_j` using an **inner** join vs. a **left** join. Explain the difference and list the affected `order_id`s.

*Your answer here.*

In [ ]:
# Your code here


*Your answer here.*

In [ ]:
# Your code here


*Your answer here.*

---
## Grading Rubric (100 pts)

| Part | Item | Pts | Full credit when… |
|---|---|---|---|
| 1 | 1.1 Session + schemas | 5 | Imports own `SparkUtils`; 4 schemas via `generate_schema`; correct types; **separate Feb schema** |
| 1 | 1.2 Read CSVs | 5 | `header` + `escape='"'`; JSON columns read intact (not split / null) |
| 1 | 1.3 Union | 5 | `unionByName`; 80 rows; O0041 sanity check passes; written answer explains positional vs. name |
| 2 | Cleaning | 15 | Drops 2 columns; `dropna(subset)` → 74; avg via `agg`; `fillna` dict; 0 nulls shown |
| 2 | Written answer | 5 | Identifies the bias zeros introduce in averages and proposes filtering `> 0` |
| 3 | 3.1 filter/count | 7 | Correct cancelled / completed / late counts and % |
| 3 | 3.2 orderBy | 8 | Correct top 5 (4 pts); outlier + two explanations (4 pts) |
| 3 | 3.3 groupBy | 10 | Completed-only filter; 3 aggregations w/ aliases & rounding; sort (6 pts); interpretation (4 pts) |
| 4 | 4.1 JSON | 10 | 8 columns, including nested paths (`location.city`, `contract.type`, `promo.code`) |
| 4 | 4.2 Joins | 10 | Two inner joins on the right keys, completed only |
| 4 | 4.3 Report + 1 action | 5 | Correct report; **exactly one action** in Part 4 (−3 if more) |
| 5 | 5.1 / 5.2 / 5.3 | 15 | 5 each — correct concepts in own words, supported by their output |